# 06 - Mantenimiento Predictivo

## Pregunta de negocio: *"¿Podemos predecir mantenimiento antes de la avería?"*

---

## Contexto y Teoría

El **mantenimiento predictivo** es el objetivo final de nuestro pipeline de datos. En lugar de reaccionar a fallos o seguir calendarios fijos, usamos datos de sensores para predecir cuándo un vehículo necesitará mantenimiento.

### Tres enfoques de mantenimiento

| Enfoque | Descripción | Costo | Riesgo |
|---------|-------------|-------|--------|
| **Reactivo** | Se repara cuando falla | Bajo (hasta que falla) | Alto: averías inesperadas, paradas no planificadas |
| **Preventivo** | Mantenimiento en intervalos fijos (cada X km, cada Y meses) | Medio: se hacen intervenciones innecesarias | Medio: no cubre fallos inesperados |
| **Predictivo** | Se interviene cuando los datos lo sugieren | Óptimo: solo cuando es necesario | Bajo: se anticipa al fallo |

### Conceptos clave

1. **Time-to-failure (TTF):** Tiempo restante hasta la próxima falla. Si lo pudiéramos medir directamente, el problema sería trivial. En la práctica, lo estimamos.

2. **Feature engineering desde sensores:** Las lecturas instantáneas de sensores no son tan útiles como sus **tendencias**. Necesitamos calcular:
   - Rolling mean/std: ¿la variable está cambiando?
   - Pendientes (slopes): ¿hacia dónde va la tendencia?
   - Conteo de anomalías: ¿hay más eventos inusuales?

3. **Análisis costo-beneficio:** No buscamos el modelo más preciso, sino el umbral de decisión que minimice el costo total:
   - **Falso positivo (FP):** Inspección innecesaria. Costo: \$200 (mano de obra + tiempo parado)
   - **Falso negativo (FN):** Fallo no detectado. Costo: \$5,000 (reparación mayor + grúa + cliente insatisfecho)
   - El costo de un FN es 25x mayor que un FP, así que preferimos modelos con alta sensibilidad.

### Nota sobre el target sintético

En un escenario real, tendríamos registros históricos de mantenimientos y fallos. Como estamos aprendiendo, **creamos un target sintético** basado en reglas razonables sobre los datos de sensores. Esto es una práctica común en proyectos educativos y de prueba de concepto.

---

## 1. Carga de datos

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

vtype_colors = {
    'electrico': '#2ecc71',
    'gasolina': '#3498db',
    'hibrido': '#9b59b6',
    'deportivo': '#e74c3c'
}

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
print(f"Raíz del proyecto: {project_root}")

In [ ]:
# Cargar telemetría completa
telemetry_files = sorted(glob.glob(os.path.join(project_root, 'data/raw/telemetry/telemetry_*.csv')))
print(f"Archivos de telemetría: {len(telemetry_files)}")

df_list = []
for f in telemetry_files:
    df_list.append(pd.read_csv(f, parse_dates=['timestamp']))
telemetry = pd.concat(df_list, ignore_index=True)

# Cargar perfiles de flota
fleet = pd.read_csv(os.path.join(project_root, 'data/raw/fleet_profiles.csv'))
telemetry = telemetry.merge(fleet[['vehicle_id', 'vehicle_type']], on='vehicle_id', how='left')

telemetry['date'] = telemetry['timestamp'].dt.date
telemetry['date'] = pd.to_datetime(telemetry['date'])

print(f"Total registros: {len(telemetry):,}")
print(f"Vehículos: {telemetry['vehicle_id'].nunique()}")
print(f"Rango: {telemetry['timestamp'].min()} a {telemetry['timestamp'].max()}")

---

## 2. Agregación de features por vehículo y día

Transformamos las lecturas de sensores a nivel de registro en features diarias por vehículo. Esto es esencial para el mantenimiento predictivo: necesitamos ver tendencias, no puntos individuales.

In [ ]:
# Variables clave de sensores
sensor_cols = ['fuel_consumption_rate', 'battery_temp_c', 'speed_kmh', 'motor_rpm', 'acceleration_ms2']
sensor_cols = [c for c in sensor_cols if c in telemetry.columns]

# Agregar por vehículo y día
agg_dict = {}
for col in sensor_cols:
    agg_dict[col] = ['mean', 'std', 'max', 'min']

daily_vehicle = telemetry.groupby(['vehicle_id', 'date', 'vehicle_type']).agg(agg_dict).reset_index()

# Aplanar columnas multi-nivel
daily_vehicle.columns = [
    f"{col[0]}_{col[1]}" if col[1] != '' else col[0]
    for col in daily_vehicle.columns
]

# Renombrar columnas vacías
daily_vehicle = daily_vehicle.rename(columns={
    'vehicle_id_': 'vehicle_id',
    'date_': 'date',
    'vehicle_type_': 'vehicle_type'
})

daily_vehicle = daily_vehicle.sort_values(['vehicle_id', 'date']).reset_index(drop=True)
print(f"Registros diarios por vehículo: {len(daily_vehicle):,}")
print(f"Columnas: {list(daily_vehicle.columns)}")
daily_vehicle.head()

---

## 3. Creación del target sintético

Definimos "necesita mantenimiento en los próximos 7 días" como nuestro target binario. La lógica se basa en señales de los sensores:

1. **Temperatura de batería** consistentemente alta (>media + 2*std del vehículo)
2. **Consumo de combustible** con tendencia creciente (pendiente positiva significativa en últimos 7 días)
3. **Frecuencia de anomalías** alta (más del 30% de las lecturas son anómalas)

Si al menos 2 de estas 3 condiciones se cumplen en los próximos 7 días, etiquetamos como `maintenance_needed = 1`.

**Importante:** Este es un target sintético para aprendizaje. En producción, usaríamos registros reales de mantenimiento.

In [ ]:
# Identificar la columna de consumo medio
consumption_col = 'fuel_consumption_rate_mean'
temp_col = 'battery_temp_c_mean' if 'battery_temp_c_mean' in daily_vehicle.columns else None
speed_col = 'speed_kmh_mean' if 'speed_kmh_mean' in daily_vehicle.columns else None

print(f"Columna de consumo: {consumption_col}")
print(f"Columna de temperatura: {temp_col}")
print(f"Columna de velocidad: {speed_col}")

# Calcular estadísticas por vehículo para definir umbrales
vehicle_stats = daily_vehicle.groupby('vehicle_id').agg({
    consumption_col: ['mean', 'std'],
}).reset_index()
vehicle_stats.columns = ['vehicle_id', 'cons_mean', 'cons_std']

if temp_col:
    temp_stats = daily_vehicle.groupby('vehicle_id')[temp_col].agg(['mean', 'std']).reset_index()
    temp_stats.columns = ['vehicle_id', 'temp_mean', 'temp_std']
    vehicle_stats = vehicle_stats.merge(temp_stats, on='vehicle_id')

print(f"\nEstadísticas por vehículo calculadas para {len(vehicle_stats)} vehículos.")

In [ ]:
# Crear features de rolling y calcular target
window = 7
maintenance_data = []

for vid, group in daily_vehicle.groupby('vehicle_id'):
    group = group.sort_values('date').copy()
    stats = vehicle_stats[vehicle_stats['vehicle_id'] == vid].iloc[0]

    # Rolling features
    group['cons_roll_mean'] = group[consumption_col].rolling(window, min_periods=3).mean()
    group['cons_roll_std'] = group[consumption_col].rolling(window, min_periods=3).std()

    if temp_col and temp_col in group.columns:
        group['temp_roll_mean'] = group[temp_col].rolling(window, min_periods=3).mean()
        group['temp_roll_std'] = group[temp_col].rolling(window, min_periods=3).std()

    # Condición 1: temperatura alta (rolling mean > media + 1.5*std)
    if temp_col and temp_col in group.columns:
        temp_threshold = stats.get('temp_mean', 30) + 1.5 * max(stats.get('temp_std', 5), 1)
        group['cond_temp_alta'] = (group['temp_roll_mean'] > temp_threshold).astype(int)
    else:
        group['cond_temp_alta'] = 0

    # Condición 2: consumo creciente (rolling mean en los próximos 7 días > actual * 1.15)
    group['cons_future_mean'] = group[consumption_col].shift(-window).rolling(window, min_periods=3).mean()
    cons_threshold = stats['cons_mean'] + 1.5 * max(stats['cons_std'], 0.01)
    group['cond_cons_alto'] = (group['cons_roll_mean'] > cons_threshold).astype(int)

    # Condición 3: variabilidad alta (rolling std > media de std * 2)
    std_threshold = max(stats['cons_std'] * 1.5, 0.01)
    group['cond_variabilidad'] = (group['cons_roll_std'] > std_threshold).astype(int)

    # Target: al menos 2 de 3 condiciones se cumplen
    group['conditions_met'] = group['cond_temp_alta'] + group['cond_cons_alto'] + group['cond_variabilidad']
    group['maintenance_needed'] = (group['conditions_met'] >= 2).astype(int)

    maintenance_data.append(group)

maint_df = pd.concat(maintenance_data, ignore_index=True)
maint_df = maint_df.dropna(subset=['cons_roll_mean'])

# Balance del target
target_counts = maint_df['maintenance_needed'].value_counts()
print("\nDistribución del target:")
print(f"  No necesita mantenimiento (0): {target_counts.get(0, 0):,} ({target_counts.get(0, 0)/len(maint_df)*100:.1f}%)")
print(f"  Necesita mantenimiento (1):    {target_counts.get(1, 0):,} ({target_counts.get(1, 0)/len(maint_df)*100:.1f}%)")
print(f"\nTotal registros con features: {len(maint_df):,}")

---

## 4. Feature engineering para mantenimiento

Construimos features más sofisticadas que capturen las tendencias temporales de cada vehículo.

In [ ]:
# Features de tendencia (pendiente de rolling mean)
for vid, group in maint_df.groupby('vehicle_id'):
    idx = group.index
    # Pendiente del consumo en los últimos 7 días
    slopes = []
    for i, row_idx in enumerate(idx):
        if i < window:
            slopes.append(0)
        else:
            window_vals = maint_df.loc[idx[i-window:i], consumption_col].values
            if len(window_vals) >= 3:
                x = np.arange(len(window_vals))
                slope, _ = np.polyfit(x, window_vals, 1)
                slopes.append(slope)
            else:
                slopes.append(0)
    maint_df.loc[idx, 'cons_slope'] = slopes

# Conteo de anomalías por IQR en ventana
for col in [consumption_col]:
    q1 = maint_df[col].quantile(0.25)
    q3 = maint_df[col].quantile(0.75)
    iqr = q3 - q1
    maint_df[f'{col}_is_outlier'] = ((maint_df[col] < q1 - 1.5*iqr) | (maint_df[col] > q3 + 1.5*iqr)).astype(int)

# Rolling count de outliers
for vid, group in maint_df.groupby('vehicle_id'):
    idx = group.index
    outlier_col = f'{consumption_col}_is_outlier'
    if outlier_col in maint_df.columns:
        maint_df.loc[idx, 'outlier_count_7d'] = maint_df.loc[idx, outlier_col].rolling(window, min_periods=1).sum().values

# Seleccionar features para el modelo
feature_candidates = [
    'cons_roll_mean', 'cons_roll_std', 'cons_slope', 'outlier_count_7d'
]

# Añadir features de las columnas originales
for col in daily_vehicle.columns:
    if col.endswith('_mean') or col.endswith('_std') or col.endswith('_max'):
        if col in maint_df.columns and col not in feature_candidates:
            feature_candidates.append(col)

# Añadir temperatura rolling si existe
if 'temp_roll_mean' in maint_df.columns:
    feature_candidates.extend(['temp_roll_mean', 'temp_roll_std'])

# Filtrar a features que realmente existen
model_features = [f for f in feature_candidates if f in maint_df.columns]
model_features = list(set(model_features))  # Eliminar duplicados

print(f"Features para el modelo ({len(model_features)}):")
for i, f in enumerate(sorted(model_features), 1):
    print(f"  {i}. {f}")

In [ ]:
# Preparar datos para modelo
model_df = maint_df[model_features + ['maintenance_needed', 'vehicle_id', 'date', 'vehicle_type']].dropna()

X = model_df[model_features]
y = model_df['maintenance_needed']

print(f"Dimensiones X: {X.shape}")
print(f"Balance del target: {y.value_counts().to_dict()}")
print(f"Tasa de positivos: {y.mean()*100:.1f}%")

# Correlación de features con el target
correlations = X.corrwith(y).sort_values(ascending=False)
print(f"\nCorrelación de features con maintenance_needed:")
for feat, corr in correlations.items():
    bar = '+' * int(abs(corr) * 30)
    sign = '+' if corr > 0 else '-'
    print(f"  {feat:<35} {corr:>7.3f} {sign}{bar}")

---

## 5. Modelos de clasificación

Entrenamos Random Forest y XGBoost (si disponible) para predecir `maintenance_needed`. Usamos split temporal (no aleatorio) y manejamos el desbalance de clases.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    confusion_matrix
)
from sklearn.preprocessing import StandardScaler

# Split temporal
model_df_sorted = model_df.sort_values('date')
split_idx = int(len(model_df_sorted) * 0.8)

train_data = model_df_sorted.iloc[:split_idx]
test_data = model_df_sorted.iloc[split_idx:]

X_train = train_data[model_features]
y_train = train_data['maintenance_needed']
X_test = test_data[model_features]
y_test = test_data['maintenance_needed']

print(f"Train: {len(X_train):,} registros ({train_data['date'].min()} a {train_data['date'].max()})")
print(f"Test:  {len(X_test):,} registros ({test_data['date'].min()} a {test_data['date'].max()})")
print(f"\nBalance train: {y_train.value_counts().to_dict()}")
print(f"Balance test:  {y_test.value_counts().to_dict()}")

# Escalar features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Calcular class_weight para manejar desbalance
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
if n_pos > 0:
    weight_ratio = n_neg / n_pos
else:
    weight_ratio = 1
print(f"Ratio de desbalance: {weight_ratio:.1f}:1")

# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train)
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]
rf_pred = rf.predict(X_test_scaled)

print("\n" + "=" * 50)
print("RANDOM FOREST")
print("=" * 50)
print(classification_report(y_test, rf_pred, target_names=['No mant.', 'Mant. necesario']))
if len(np.unique(y_test)) > 1:
    print(f"ROC-AUC: {roc_auc_score(y_test, rf_proba):.4f}")

In [ ]:
# --- XGBoost (si disponible) ---
try:
    from xgboost import XGBClassifier
    xgb_available = True
    print("XGBoost disponible.")

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        scale_pos_weight=weight_ratio,
        learning_rate=0.1,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    xgb.fit(X_train_scaled, y_train)
    xgb_proba = xgb.predict_proba(X_test_scaled)[:, 1]
    xgb_pred = xgb.predict(X_test_scaled)

    print("\n" + "=" * 50)
    print("XGBOOST")
    print("=" * 50)
    print(classification_report(y_test, xgb_pred, target_names=['No mant.', 'Mant. necesario']))
    if len(np.unique(y_test)) > 1:
        print(f"ROC-AUC: {roc_auc_score(y_test, xgb_proba):.4f}")

except ImportError:
    xgb_available = False
    print("XGBoost no disponible. Continuando solo con Random Forest.")

In [ ]:
# Curvas ROC y Precision-Recall
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if len(np.unique(y_test)) > 1:
    # ROC Curve
    fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
    auc_rf = roc_auc_score(y_test, rf_proba)
    axes[0].plot(fpr_rf, tpr_rf, color='#3498db', linewidth=2, label=f'Random Forest (AUC={auc_rf:.3f})')

    if xgb_available:
        fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_proba)
        auc_xgb = roc_auc_score(y_test, xgb_proba)
        axes[0].plot(fpr_xgb, tpr_xgb, color='#e74c3c', linewidth=2, label=f'XGBoost (AUC={auc_xgb:.3f})')

    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Aleatorio')
    axes[0].set_title('Curva ROC', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].legend()

    # Precision-Recall Curve
    prec_rf, rec_rf, _ = precision_recall_curve(y_test, rf_proba)
    ap_rf = average_precision_score(y_test, rf_proba)
    axes[1].plot(rec_rf, prec_rf, color='#3498db', linewidth=2, label=f'Random Forest (AP={ap_rf:.3f})')

    if xgb_available:
        prec_xgb, rec_xgb, _ = precision_recall_curve(y_test, xgb_proba)
        ap_xgb = average_precision_score(y_test, xgb_proba)
        axes[1].plot(rec_xgb, prec_xgb, color='#e74c3c', linewidth=2, label=f'XGBoost (AP={ap_xgb:.3f})')

    baseline = y_test.mean()
    axes[1].axhline(y=baseline, color='gray', linestyle='--', alpha=0.5, label=f'Baseline ({baseline:.3f})')
    axes[1].set_title('Curva Precision-Recall', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].legend()
else:
    axes[0].text(0.5, 0.5, 'Solo una clase en test set.\nNo se puede calcular ROC.',
                transform=axes[0].transAxes, ha='center', va='center', fontsize=14)
    axes[1].text(0.5, 0.5, 'Solo una clase en test set.\nNo se puede calcular PR.',
                transform=axes[1].transAxes, ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.show()

---

## 6. Análisis costo-beneficio

El umbral de decisión por defecto es 0.5 (probabilidad > 50% = mantenimiento). Pero dado que un falso negativo (fallo no detectado) cuesta 25x más que un falso positivo (inspección innecesaria), necesitamos encontrar el umbral óptimo que minimice el costo total esperado.

| Decisión | Realidad | Resultado | Costo |
|----------|----------|-----------|-------|
| No mantener | No falla | Verdadero Negativo | $0 |
| Mantener | Iba a fallar | Verdadero Positivo | $200 (inspección) |
| Mantener | No iba a fallar | Falso Positivo | $200 (inspección innecesaria) |
| No mantener | Falla | Falso Negativo | $5,000 (avería) |

In [ ]:
# Definir costos
COST_FP = 200    # Inspección innecesaria
COST_FN = 5000   # Fallo no detectado
COST_TP = 200    # Inspección correcta (se previene el fallo)
COST_TN = 0      # No hacer nada correctamente

# Elegir el mejor modelo
if xgb_available and len(np.unique(y_test)) > 1:
    best_proba = xgb_proba if roc_auc_score(y_test, xgb_proba) > roc_auc_score(y_test, rf_proba) else rf_proba
    best_name = 'XGBoost' if roc_auc_score(y_test, xgb_proba) > roc_auc_score(y_test, rf_proba) else 'Random Forest'
else:
    best_proba = rf_proba
    best_name = 'Random Forest'

# Calcular costo para distintos umbrales
thresholds = np.arange(0.05, 0.96, 0.01)
costs = []

for thresh in thresholds:
    preds = (best_proba >= thresh).astype(int)
    cm = confusion_matrix(y_test, preds, labels=[0, 1])

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        # Si solo hay una clase
        tn = cm[0, 0] if cm.shape[0] > 0 else 0
        fp = fn = tp = 0

    total_cost = (fp * COST_FP) + (fn * COST_FN) + (tp * COST_TP) + (tn * COST_TN)
    avg_cost = total_cost / len(y_test)
    costs.append({
        'threshold': thresh,
        'total_cost': total_cost,
        'avg_cost': avg_cost,
        'fp': fp, 'fn': fn, 'tp': tp, 'tn': tn
    })

costs_df = pd.DataFrame(costs)

# Umbral óptimo
optimal_idx = costs_df['total_cost'].idxmin()
optimal_threshold = costs_df.loc[optimal_idx, 'threshold']
optimal_cost = costs_df.loc[optimal_idx, 'total_cost']
default_cost = costs_df[costs_df['threshold'].round(2) == 0.50]['total_cost'].values
default_cost = default_cost[0] if len(default_cost) > 0 else costs_df.iloc[len(costs_df)//2]['total_cost']

print(f"Modelo: {best_name}")
print(f"\nUmbral por defecto (0.50): Costo total = ${default_cost:,.0f}")
print(f"Umbral óptimo ({optimal_threshold:.2f}): Costo total = ${optimal_cost:,.0f}")
print(f"Ahorro: ${default_cost - optimal_cost:,.0f} ({(default_cost - optimal_cost)/default_cost*100:.1f}%)")

In [ ]:
# Visualizar costo vs umbral
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Costo total vs umbral
axes[0].plot(costs_df['threshold'], costs_df['total_cost'], color='#3498db', linewidth=2)
axes[0].axvline(x=optimal_threshold, color='#2ecc71', linestyle='--', linewidth=2, label=f'Óptimo: {optimal_threshold:.2f}')
axes[0].axvline(x=0.5, color='#e74c3c', linestyle=':', linewidth=2, label='Default: 0.50')
axes[0].scatter([optimal_threshold], [optimal_cost], color='#2ecc71', s=100, zorder=5)
axes[0].set_title('Costo total vs umbral de decisión', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Umbral de probabilidad')
axes[0].set_ylabel('Costo total ($)')
axes[0].legend(fontsize=11)

# FP y FN vs umbral
axes[1].plot(costs_df['threshold'], costs_df['fp'] * COST_FP, color='#f39c12', linewidth=2, label=f'Costo FP (${COST_FP}/caso)')
axes[1].plot(costs_df['threshold'], costs_df['fn'] * COST_FN, color='#e74c3c', linewidth=2, label=f'Costo FN (${COST_FN}/caso)')
axes[1].plot(costs_df['threshold'], costs_df['total_cost'], color='#3498db', linewidth=2, linestyle='--', label='Costo total')
axes[1].axvline(x=optimal_threshold, color='#2ecc71', linestyle='--', alpha=0.7)
axes[1].set_title('Descomposición de costos por tipo de error', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Umbral de probabilidad')
axes[1].set_ylabel('Costo ($)')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

# Matriz de confusión con umbral óptimo
optimal_preds = (best_proba >= optimal_threshold).astype(int)
cm_optimal = confusion_matrix(y_test, optimal_preds, labels=[0, 1])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm_optimal, annot=True, fmt='d', cmap='Blues',
    xticklabels=['No mant.', 'Mant. necesario'],
    yticklabels=['No mant.', 'Mant. necesario'],
    ax=ax
)
ax.set_title(f'Matriz de confusión (umbral={optimal_threshold:.2f})', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
plt.tight_layout()
plt.show()

---

## 7. Dashboard de priorización de mantenimiento

Para cada vehículo, mostramos la probabilidad predicha de necesitar mantenimiento a lo largo del tiempo. Esto permite al equipo de operaciones priorizar qué vehículos atender primero.

In [ ]:
# Predicción sobre todos los datos con el mejor modelo
all_features_scaled = scaler.transform(model_df_sorted[model_features])

if xgb_available and best_name == 'XGBoost':
    all_proba = xgb.predict_proba(all_features_scaled)[:, 1]
else:
    all_proba = rf.predict_proba(all_features_scaled)[:, 1]

model_df_sorted = model_df_sorted.copy()
model_df_sorted['maint_probability'] = all_proba

# Última probabilidad por vehículo (estado actual)
latest_status = (
    model_df_sorted
    .sort_values('date')
    .groupby(['vehicle_id', 'vehicle_type'])
    .last()
    .reset_index()
    [['vehicle_id', 'vehicle_type', 'maint_probability', 'date']]
    .sort_values('maint_probability', ascending=False)
)

# Clasificar prioridad
def priority_label(prob, threshold=optimal_threshold):
    if prob >= threshold * 1.5:
        return 'URGENTE'
    elif prob >= threshold:
        return 'PROGRAMAR'
    elif prob >= threshold * 0.5:
        return 'MONITOREAR'
    else:
        return 'OK'

latest_status['prioridad'] = latest_status['maint_probability'].apply(priority_label)

print("DASHBOARD DE PRIORIZACIÓN DE MANTENIMIENTO")
print("=" * 70)
print(f"{'Vehículo':<15} {'Tipo':<12} {'Probabilidad':>12} {'Prioridad':<12}")
print("-" * 70)
for _, row in latest_status.iterrows():
    prob_str = f"{row['maint_probability']*100:.1f}%"
    print(f"{row['vehicle_id']:<15} {row['vehicle_type']:<12} {prob_str:>12} {row['prioridad']:<12}")
print("=" * 70)

priority_summary = latest_status['prioridad'].value_counts()
print(f"\nResumen: {priority_summary.to_dict()}")

In [ ]:
# Visualización del dashboard
fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(latest_status) * 0.35)))

# 1. Barras horizontales de probabilidad por vehículo
priority_colors_map = {
    'URGENTE': '#e74c3c',
    'PROGRAMAR': '#f39c12',
    'MONITOREAR': '#3498db',
    'OK': '#2ecc71'
}
bar_colors = [priority_colors_map.get(p, '#95a5a6') for p in latest_status['prioridad']]

axes[0].barh(
    range(len(latest_status)),
    latest_status['maint_probability'] * 100,
    color=bar_colors, edgecolor='white', linewidth=0.5
)
axes[0].set_yticks(range(len(latest_status)))
axes[0].set_yticklabels([f"{row['vehicle_id']} ({row['vehicle_type']})" for _, row in latest_status.iterrows()], fontsize=9)
axes[0].axvline(x=optimal_threshold * 100, color='gray', linestyle='--', alpha=0.7, label=f'Umbral: {optimal_threshold*100:.0f}%')
axes[0].set_xlabel('Probabilidad de mantenimiento (%)')
axes[0].set_title('Priorización de mantenimiento por vehículo', fontsize=13, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].invert_yaxis()

# 2. Evolución temporal de probabilidad para top 5 vehículos
top5_vehicles = latest_status.head(5)['vehicle_id'].values

for vid in top5_vehicles:
    vdata = model_df_sorted[model_df_sorted['vehicle_id'] == vid].sort_values('date')
    vtype = vdata['vehicle_type'].iloc[0]
    color = vtype_colors.get(vtype, '#34495e')
    axes[1].plot(vdata['date'], vdata['maint_probability'] * 100,
               linewidth=2, label=f'{vid} ({vtype})', color=color, alpha=0.8)

axes[1].axhline(y=optimal_threshold * 100, color='gray', linestyle='--', alpha=0.7, label='Umbral')
axes[1].set_title('Evolución de probabilidad de mantenimiento (Top 5)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Fecha')
axes[1].set_ylabel('Probabilidad (%)')
axes[1].legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance del mejor modelo
if best_name == 'XGBoost' and xgb_available:
    importances = xgb.feature_importances_
else:
    importances = rf.feature_importances_

feat_imp = pd.DataFrame({
    'feature': model_features,
    'importance': importances
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(5, len(feat_imp) * 0.35)))
ax.barh(feat_imp['feature'], feat_imp['importance'], color='#3498db', edgecolor='white')
ax.set_title(f'Importancia de features ({best_name})', fontsize=13, fontweight='bold')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

print("\nTop 5 features más importantes:")
for _, row in feat_imp.tail(5).iloc[::-1].iterrows():
    print(f"  {row['feature']:<35} {row['importance']:.4f}")

---

## 8. Resumen: Fase 5 completa

### Pipeline completo: de datos crudos a decisiones de mantenimiento

```
Datos de sensores (telemetría)
    |
    v
[04] Forecasting --> Proyección de costos de combustible
    |                  ARIMA, baselines, forecast por vehículo
    v
[05] Anomalías --> Detección automática de problemas
    |                  Z-score, IQR, Isolation Forest, drift
    v
[06] Mantenimiento --> Predicción de cuándo intervenir
    |                  RF/XGBoost, análisis costo-beneficio
    v
Dashboard de priorización --> Decisión operativa
```

### Logros de la Fase 5

1. **Forecasting (Notebook 04):** Comparamos 4 métodos de pronóstico de consumo de combustible. Aprendimos a usar ACF/PACF para seleccionar parámetros ARIMA y a nunca hacer split aleatorio en series temporales.

2. **Detección de anomalías (Notebook 05):** Implementamos un sistema de 3 capas (estadístico, Isolation Forest, drift) que puede funcionar en tiempo real, batch diario y análisis semanal.

3. **Mantenimiento predictivo (Notebook 06):** Construimos un modelo que predice la necesidad de mantenimiento y optimizamos su umbral de decisión considerando los costos reales de falsos positivos vs falsos negativos.

### Impacto de negocio estimado

- **Reducción de averías inesperadas** al detectar degradación antes de que cause fallos.
- **Optimización de costos de inspección** al intervenir solo cuando el modelo lo indica.
- **Mejor planificación** de presupuestos operativos con forecasting de consumo.

### Próximos pasos (Fase 6)

La Fase 6 se centrará en la presentación de resultados y la creación de dashboards interactivos para comunicar estos hallazgos a stakeholders no técnicos.